In [2]:
import os, json, yaml, time
import datetime
import urllib3
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Proj
from scipy.spatial import cKDTree
from itertools import product
from scipy.stats import bootstrap, ecdf
from shapely.geometry import Point, MultiPolygon
import multiprocessing
from functools import partial
from multiprocessing import Pool


### List and verify metrics files

In [3]:
base_data_path = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121/"
start_date = "2023-03-15"
end_date = "2024-04-29"

date_range = pd.date_range(start=start_date, end=end_date, freq='d')
data_files = date_range.strftime(f"{base_data_path}veraset_%Y-%m-%d_PST_chunked.parquet")
files = []
for i, f in enumerate(data_files):
    if os.path.isfile(f):
        files.append(f)
print(len(files))

92


### Create boolean columns for FFWs and rain amounts during the AR period

#### FFWs

In [14]:
def get_bool(row, polys):
    point = Point(row[["lng_home", "lat_home"]])
    return any([p.contains(point) for p in polys])

In [15]:
for file in files[58:64]:
    df_temp = pd.read_parquet(file)
    s_date = file.split("_PST")[0].split("veraset_")[1]
    e_date = (datetime.datetime.strptime(s_date, '%Y-%m-%d') + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
    url = f"https://mesonet.agron.iastate.edu/api/1/cow.json?wfo=LOX&begints={s_date}T00:00:00Z&enable_shared_border=1&endts={e_date}T00:00:00Z&hailsize=1&wind=58&phenomena=FF&lsrtype=FF&lsrbuffer=30&warningbuffer=1"
    req = urllib3.request("GET", url)
    ffw_warnings = json.loads(req.data)["events"]["features"]
    polygons = [MultiPolygon(w["geometry"]["coordinates"]) for w in ffw_warnings]
    df_temp[f"FFW"] = df_temp.apply(get_bool, polys=polygons, axis=1)
    df_temp.to_parquet(file.replace("analysis_ready_data_20251121", "analysis_ready_data_20251121_metavars"))

In [17]:
files_updated = [f.replace("analysis_ready_data_20251121", "analysis_ready_data_20251121_metavars") for f in files]

#### MRMS Variables

In [18]:
def create_tree(mrms_ds):
    # define projection
    proj = Proj(proj="lcc", R=6371229, lat_0=25, lon_0=265, lat_1=25, lat_2=25)
    
    # grid and project MRMS lat-lon coords then create KDTree
    mrms_lon_grid = np.tile(mrms_ds['lon'].values-360, (mrms_ds['lat'].values.shape[0], 1)).T
    mrms_lat_grid = np.tile(mrms_ds['lat'].values, (mrms_ds['lon'].values.shape[0], 1))
    proj_mrms_lon_grid, proj_mrms_lat_grid = np.array(proj(mrms_lon_grid, mrms_lat_grid))
    tree = cKDTree(np.c_[proj_mrms_lon_grid.ravel(), proj_mrms_lat_grid.ravel()])
    return tree, proj, mrms_lon_grid.shape

def get_mrms_var(df_temp, date, idx, mrms_ds, variable, variable_name):

    # grab MRMS data using calculated index
    lon_idx_da = xr.DataArray(idx[:, 0], dims="z")
    lat_idx_da = xr.DataArray(idx[:, 1], dims="z")
    mrms_idx = mrms_ds[variable].sel(time=date).isel(lon=lon_idx_da, lat=lat_idx_da).values
    series = pd.Series(mrms_idx.tolist(), name=variable_name, index=df_temp.index)
    return series

def cycle_through_mrms_variables(file, variables, variable_aggregates):

    mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variables[0]}_2024-03-15T00:00_2024-04-30T00:00_daily_{variable_aggregates[0]}.nc"
    mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
    tree, proj, mrms_lon_grid_shape = create_tree(mrms_ds)
    
    df_temp = pd.read_parquet(file)
    date = df_temp["date"].iloc[0].to_numpy()
    year = df_temp["date"].iloc[0].year
    _, points = tree.query(np.column_stack(proj(df_temp['lng_home'], df_temp['lat_home'])))
    idx = np.column_stack(np.unravel_index(points, mrms_lon_grid_shape))

    for variable, variable_aggregate in product(variables, variable_aggregates):
        mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variable}_{year}-03-15T00:00_{year}-04-30T00:00_daily_{variable_aggregate}.nc"
        mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
        variable_name = f"streamflow_{variable_aggregate}" if "STREAMFLOW" in variable else f"precip_{variable_aggregate}"
        df_temp[f"{variable_name}"] = get_mrms_var(df_temp, date, idx, mrms_ds, variable, variable_name)
    df_temp.to_parquet(file)
    
    return None

In [19]:
variables = ["MultiSensor_QPE_01H_Pass2_00.00", "FLASH_HP_MAXUNITSTREAMFLOW_00.00"]
variable_aggregates = ["max", "sum"]

n_processors = multiprocessing.cpu_count()
f = partial(cycle_through_mrms_variables,
            variables=variables,
            variable_aggregates=variable_aggregates)

s = time.time()
print('Beginning data processing...')
with Pool(processes=n_processors) as pool:
    result = pool.map_async(f, files_updated[58:64])
    results = result.get()
print(f"{np.round((time.time() - s) / 60, 2)} minutes to complete parallel computation.")
print(len(results))

Beginning data processing...
0.17 minutes to complete parallel computation.
6


In [20]:
df_temp = pd.read_parquet(files_updated[58])
df_temp

,uid,duration,lat_home,lng_home,zipcode,row_group,county,number_of_visits,distance_straight_line,jump_lengths,radius_of_gyration,num_stops,max_distance_from_home,date,FFW,precip_max,precip_sum,streamflow_max,streamflow_sum
10,0000d5706648bfddd64ea083ea418ee3804ab9a3a89ed4...,2 days 09:09:58,34.157609,-118.267499,91203,0,LA,8,0.043692,"[0.006536631742133652, 0.0022340374554451156, ...",0.005340,1,0.008097,2024-03-27,False,0.0,0.0,0.0,0.0
64,0000d3f70d721e99161e7f39154bff9d4319a53f6baad7...,4 days 23:15:49,34.107017,-118.200225,90042,0,LA,18,0.105365,"[0.018293535188910033, 0.008936582874833126, 0...",0.005239,1,2.594688,2024-03-27,False,0.0,0.0,0.0,0.0
255,0000a3d66479e9b9fd41b203a3ff56844d374fb8e06020...,1 days 04:40:49,34.080940,-118.144508,91803,0,LA,6,3.775118,"[3.7584601701937244, 0.004991906022544699, 0.0...",1.399698,1,56.667315,2024-03-27,False,0.0,0.0,0.0,0.0
273,0000a296f1765ec0c320794528ca5aff11172a684c6971...,1 days 20:20:32,34.679140,-118.108208,93535,0,LA,41,0.265014,"[0.014141826940279234, 0.009780938835067863, 0...",0.005556,1,0.435139,2024-03-27,False,0.0,0.0,0.0,0.0
333,000093ca81930e1c247112fceaa58e2d2e3df3e1d438f6...,21 days 11:29:35,34.099681,-117.846496,91724,0,LA,13,4.462821,"[0.0067867997217267995, 0.3569048383976517, 0....",1.615193,2,3.587737,2024-03-27,False,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ffffd6b822649917a6b3816950e4294172d718228f8c86...,1 days 09:50:08,34.267865,-118.660888,93063,75017,VE,5,53.228476,"[0.016039068925148354, 0.00697970070570399, 0....",21.280961,1,52.219462,2024-03-27,False,0.0,0.0,0.0,0.0
235,ffffca96377393d66fe59124ed2ed3ab1ba82589eed758...,1 days 03:27:45,34.092821,-118.305691,90029,75017,LA,2,0.000000,[0.0],0.000000,1,0.005285,2024-03-27,False,0.0,0.0,0.0,0.0
372,ffffa5a7e8c0b8e6d98a832b851277bc3279115353ba87...,4 days 11:43:54,33.892442,-118.358534,90260,75017,LA,38,0.205982,"[0.006401476058383103, 0.002466259658623886, 0...",0.008263,1,0.025402,2024-03-27,False,0.0,0.0,0.0,0.0
476,ffff9185c3f5cffe3f30f6274e19793928874bfcb98e9e...,2 days 19:02:52,34.057878,-117.932276,91790,75017,LA,6,0.034510,"[0.010674669394046524, 0.007145075302993636, 0...",0.004041,1,3.359770,2024-03-27,False,0.0,0.0,0.0,0.0


In [21]:
AR_dates = pd.date_range("2024-03-29", "2024-04-01", freq="1d")

bool_series = []
for date in AR_dates:
    df_temp = pd.read_parquet(files_updated[58].replace("2023-03-27", date.strftime('%Y-%m-%d')))
    bool_series.append(df_temp["FFW"].rename(f"FFW_{date.strftime('%Y-%m-%d')}"))
    for mrms_metric in ['precip_max', 'precip_sum', 'streamflow_max', 'streamflow_sum']:
        bool_series.append(df_temp.apply(lambda x: True if x[mrms_metric] > 0 else False, axis=1).rename(f"{mrms_metric}_{date.strftime('%Y-%m-%d')}"))
bool_df = pd.concat(bool_series, axis=1)

In [23]:
bool_df.shape

(790763, 20)

In [24]:
for file in files_updated[58:64]:
    df_temp = pd.read_parquet(file)
    print(df_temp.shape)
    df_temp = pd.merge(df_temp, bool_df, left_index=True, right_index=True, how='left')
    print(df_temp.shape)
df_temp

(790763, 19)
(601004813, 39)
(915525, 19)


KeyboardInterrupt: 

In [26]:
df_temp.shape

(915525, 19)

In [25]:
df_temp

,uid,duration,lat_home,lng_home,zipcode,row_group,county,number_of_visits,distance_straight_line,jump_lengths,radius_of_gyration,num_stops,max_distance_from_home,date,FFW,precip_max,precip_sum,streamflow_max,streamflow_sum
65,0000d3f70d721e99161e7f39154bff9d4319a53f6baad7...,4 days 23:15:49,34.107017,-118.200225,90042,0,LA,23,40.064967,"[0.0036408734713914326, 0.007470399476989095, ...",6.974641,4,14.249769,2024-03-28,False,0.0,0.0,0.0,0.0
151,0000c4f1622a043a14ff5ab51aed9833409c8867f227aa...,3 days 19:46:29,33.799465,-118.208737,90810,0,LA,24,14.536413,"[0.05436925637708918, 0.0055591718235304165, 0...",4.571014,1,12.748204,2024-03-28,False,0.0,0.0,0.0,0.0
274,0000a296f1765ec0c320794528ca5aff11172a684c6971...,1 days 20:20:32,34.679140,-118.108208,93535,0,LA,10,57.018401,"[0.0010983631486676743, 0.0042987376460246985,...",27.526971,1,56.794834,2024-03-28,False,0.0,0.0,0.0,0.0
334,000093ca81930e1c247112fceaa58e2d2e3df3e1d438f6...,21 days 11:29:35,34.099681,-117.846496,91724,0,LA,8,0.121257,"[0.0057443160599120625, 0.0359038057279151, 0....",0.016693,1,0.024352,2024-03-28,False,0.0,0.0,0.0,0.0
494,000064e1184fc9ba45ee54f9feee9ed1c25244525facbc...,4 days 19:08:15,34.147367,-118.344499,91505,0,LA,46,2.279246,"[0.005938449756687779, 0.0048726364608435435, ...",0.163053,3,0.505700,2024-03-28,False,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,ffffa5a7e8c0b8e6d98a832b851277bc3279115353ba87...,4 days 11:43:54,33.892442,-118.358534,90260,75017,LA,73,221.315143,"[0.006401476079172034, 0.006362624739560951, 0...",38.898213,5,90.135500,2024-03-28,False,0.0,0.0,0.0,0.0
499,ffff8c6eca66e179685ada7106011ec94a8e64faf7aa0f...,4 days 11:32:52,34.033876,-118.166443,90022,75017,LA,7,0.033083,"[0.0012725249479121903, 0.00551427477381461, 0...",0.006397,1,0.010367,2024-03-28,False,0.0,0.0,0.0,0.0
570,ffff88da7942d1c71e8cc139e95c8ae3d0b36061725b3b...,5 days 14:43:57,33.907301,-118.164314,90723,75017,LA,8,6.157200,"[6.113982933808087, 0.0025450498958243806, 0.0...",2.022718,2,5.754233,2024-03-28,False,0.0,0.0,0.0,0.0
628,ffff77ac0ade8f1cc4e6abe877aa97becc7adf1fed88c1...,2 days 19:03:58,33.999302,-118.286805,90037,75017,LA,6,0.014888,"[0.004603858061351565, 0.002992428338943719, 0...",0.002172,1,4.858646,2024-03-28,False,0.0,0.0,0.0,0.0


### Calculate Bootsrap average, sign, and p-value

In [ ]:
for mrms_label in ["precip", "streamflow"]:
    mrms_metrics = [c for c in list(df.columns) if any(k in c for k in [f'{mrms_label}_max_', f'{mrms_label}_sum_'])]
    mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
    mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
    deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
    counties = ["LA", "SB", "VE"]
    bools = ["yes", "no"]
    
    for county in counties:
        df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
        for mrms_metric in mrms_metrics:
            for delta in deltas:
                for bool in bools:
                    data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                    if data.shape[0]<2:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                        continue
                    else:
                        res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                        res_ecdf = ecdf(res.bootstrap_distribution)
                        res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                        if res_ecdf_0 > 0.5:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                        else:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
        df_bs.to_parquet(f"bootstrapping_daily/results_{mrms_label}_{county}.parquet")

In [ ]:
mrms_metrics = [c for c in list(df.columns) if "_ffw" in c]
mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
counties = ["LA", "SB", "VE"]
bools = ["yes", "no"]

for county in counties:
    df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
    for mrms_metric in mrms_metrics:
        for delta in deltas:
            for bool in bools:
                data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                if data.shape[0]<2:
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                    continue
                else:
                    res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                    res_ecdf = ecdf(res.bootstrap_distribution)
                    res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                    if res_ecdf_0 > 0.5:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                    else:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
    df_bs.to_parquet(f"bootstrapping_daily/results_ffw_{county}.parquet")